# Four baselines on the corrected MOOCCubeX chronological splits

This notebook trains **NARM, BERT4Rec, LightGCN, and HGT**. SASRec is intentionally excluded.

The notebook:

- reads the same `processed/splits/train.parquet`, `valid.parquet`, and `test.parquet` used by BCE-SASRec;
- restricts candidates to the training catalogue;
- verifies chronological order and catalogue coverage before training;
- prevents validation/test targets from entering training-time negative sampling;
- selects each checkpoint using **validation NDCG@10**;
- evaluates the untouched test targets under full-catalogue ranking;
- runs seeds **42, 2026, and 3407** and exports per-user ranks for paired significance tests.

Use a GPU runtime. In Kaggle, attach the processed dataset using **Add Input**. In Colab, mount Drive or set `PROCESSED_DIR` explicitly.

In [ ]:
# Runtime setup: mount Drive and locate the EXACT paper dataset.
from pathlib import Path
import os
import pandas as pd
import zipfile

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/BCE_SASRec_Four_Baselines_Results'
else:
    OUTPUT_DIR = '/kaggle/working/four_baselines_correct' if Path('/kaggle/working').exists() else './four_baselines_correct'

EXPECTED = (152022, 17959, 4720)  # train rows, common evaluation users, train items

def find_exact_processed_dataset():
    roots = []
    if IN_COLAB:
        roots.append(Path('/content/drive/MyDrive'))
        # If the full processed dataset was uploaded as a ZIP, extract it to the
        # fast Colab runtime disk and include the extracted tree in the search.
        zip_candidates = list(Path('/content/drive/MyDrive').rglob('MOOCCubeX_processed.zip'))
        if zip_candidates:
            archive = max(zip_candidates, key=lambda p: p.stat().st_mtime)
            extract_root = Path('/content/MOOCCubeX_processed_full')
            marker = extract_root/'.extraction_complete'
            if not marker.exists() or marker.stat().st_mtime < archive.stat().st_mtime:
                extract_root.mkdir(parents=True, exist_ok=True)
                print('Extracting full processed dataset:', archive)
                with zipfile.ZipFile(archive, 'r') as zf:
                    zf.extractall(extract_root)
                marker.touch()
                print('Extraction complete:', extract_root)
            else:
                print('Using previously extracted archive:', extract_root)
            roots.insert(0, extract_root)
    if Path('/kaggle/input').exists():
        roots.append(Path('/kaggle/input'))
    roots.extend([Path('/content'), Path('.')])
    checked = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for train_file in root.rglob('train.parquet'):
            split_dir = train_file.parent
            processed = split_dir.parent
            key = str(processed.resolve())
            if key in seen or not (split_dir/'valid.parquet').exists() or not (split_dir/'test.parquet').exists():
                continue
            seen.add(key)
            try:
                train = pd.read_parquet(train_file, columns=['user_id','video_id'])
                valid = pd.read_parquet(split_dir/'valid.parquet', columns=['user_id'])
                test = pd.read_parquet(split_dir/'test.parquet', columns=['user_id'])
                common_users = len(set(train.user_id.astype(str)) & set(valid.user_id.astype(str)) & set(test.user_id.astype(str)))
                stats = (len(train), common_users, train.video_id.astype(str).nunique())
                checked.append((str(processed), stats))
                print('Candidate:', processed, '->', stats)
                if stats == EXPECTED:
                    graph = processed/'graph'
                    required_graph = ['video_index.parquet','concept_video_edges.parquet','course_video_edges.parquet']
                    missing = [x for x in required_graph if not (graph/x).exists()]
                    if missing:
                        print('  Counts match, but graph files are missing:', missing)
                        continue
                    return str(processed)
            except Exception as exc:
                print('Skipped', processed, 'because:', exc)
    details = '\n'.join(f'  {path}: {stats}' for path,stats in checked) or '  No complete split folders were found.'
    raise FileNotFoundError(
        'The full 17,959-user processed dataset is not available yet. Wait for '
        'MOOCCubeX_processed.zip to finish uploading, then rerun this cell.\n'
        f'Expected train/common-users/train-items = {EXPECTED}.\nFound:\n' + details
    )

PROCESSED_DIR = find_exact_processed_dataset()

print('Colab runtime:', IN_COLAB)
print('Selected processed directory:', PROCESSED_DIR)
print('Output directory:', OUTPUT_DIR)


In [ ]:
from __future__ import annotations
import copy, gc, json, math, os, random, time, warnings
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

@dataclass
class Config:
    seeds: tuple = (42, 2026, 3407)
    models: tuple = ("NARM", "BERT4Rec", "LightGCN", "HGT")
    max_len: int = 50
    hidden_dim: int = 128
    layers: int = 2
    heads: int = 4
    dropout: float = 0.10
    batch_size: int = 256
    eval_batch_size: int = 256
    negatives: int = 100
    max_epochs: int = 50
    minimum_epochs: int = 8
    patience: int = 4
    min_delta: float = 1e-4
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    gradient_clip: float = 1.0
    max_concepts_per_video: int = 12
    ks: tuple = (5, 10, 20)
    num_workers: int = 2
    strict_expected_counts: bool = True
    expected_train_interactions: int = 152022
    expected_evaluation_users: int = 17959
    expected_training_catalogue: int = 4720

CFG = Config()

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def locate_processed(explicit=None):
    candidates = [
        explicit, os.getenv("BCE_PROCESSED"),
        "/kaggle/working/processed",
        "/kaggle/input/datasets/kabil908/processed/processed",
        "/content/drive/MyDrive/DataCon/processed",
        "./processed",
    ]
    for value in candidates:
        if value:
            p = Path(value)
            if all((p / "splits" / f).exists() for f in ("train.parquet", "valid.parquet", "test.parquet")):
                return p
    for root in (Path("/kaggle/input"), Path("/content/drive/MyDrive"), Path(".")):
        if root.exists():
            for f in root.rglob("train.parquet"):
                p = f.parent.parent
                if (p/"splits"/"valid.parquet").exists() and (p/"splits"/"test.parquet").exists():
                    return p
    raise FileNotFoundError("Processed dataset not found. Set PROCESSED_DIR to the folder containing splits/ and graph/.")

def left_pad(values, n, pad=0):
    values = list(values)[-n:]
    return [pad] * (n - len(values)) + values

def read_splits(processed, cfg):
    split_dir, graph_dir = processed/"splits", processed/"graph"
    frames = [pd.read_parquet(split_dir/f).sort_values(["user_id", "timestamp"]).reset_index(drop=True)
              for f in ("train.parquet", "valid.parquet", "test.parquet")]
    train_df, valid_df, test_df = frames
    required = {"user_id", "video_id", "timestamp"}
    for name, frame in zip(("train", "validation", "test"), frames):
        missing = required - set(frame.columns)
        if missing: raise ValueError(f"{name} is missing columns: {sorted(missing)}")
        frame["user_id"] = frame.user_id.astype(str)
        frame["video_id"] = frame.video_id.astype(str)
        frame["timestamp"] = pd.to_numeric(frame.timestamp, errors="coerce").fillna(0).astype("int64")

    # The training catalogue is the only eligible item catalogue.
    item_ids = sorted(train_df.video_id.unique())
    user_ids = sorted(set(train_df.user_id) | set(valid_df.user_id) | set(test_df.user_id))
    item2idx = {x:i+1 for i,x in enumerate(item_ids)}
    user2idx = {x:i for i,x in enumerate(user_ids)}
    def mapped(frame):
        x = frame[frame.video_id.isin(item2idx)].copy().reset_index(drop=True)
        x["u"] = x.user_id.map(user2idx).astype("int64")
        x["i"] = x.video_id.map(item2idx).astype("int64")
        return x
    train, valid, test = map(mapped, frames)

    def history(frame):
        return {int(u):g.i.astype(int).tolist() for u,g in frame.groupby("u", sort=False)}
    train_h, valid_only, test_only = map(history, (train, valid, test))
    eval_users = sorted(set(train_h) & set(valid_only) & set(test_only))
    valid_h = {u:list(train_h[u]) for u in eval_users}
    test_h = {u:list(train_h[u]) + [valid_only[u][0]] for u in eval_users}
    valid_target = {u:valid_only[u][0] for u in eval_users}
    test_target = {u:test_only[u][0] for u in eval_users}

    train_max = train.groupby("u").timestamp.max()
    vrow, trow = valid.set_index("u"), test.set_index("u")
    audits = {
        "validation_is_one_target_per_user": bool(valid.groupby("u").size().eq(1).all()),
        "test_is_one_target_per_user": bool(test.groupby("u").size().eq(1).all()),
        "train_before_validation": bool((train_max.loc[eval_users].values <= vrow.loc[eval_users].timestamp.values).all()),
        "validation_before_test": bool((vrow.loc[eval_users].timestamp.values <= trow.loc[eval_users].timestamp.values).all()),
        "validation_in_training_catalogue": bool(valid.i.isin(set(train.i)).all()),
        "test_in_training_catalogue": bool(test.i.isin(set(train.i)).all()),
    }
    if not all(audits.values()): raise RuntimeError(f"Split/leakage audit failed: {audits}")

    # Concept matrix is used only for Concept Recall evaluation and HGT graph relations.
    n_items = len(item_ids)
    item_concepts = np.zeros((n_items+1, cfg.max_concepts_per_video), np.int64)
    concepts, courses = [], []
    item_course = np.zeros(n_items+1, np.int64)
    vi_path, cv_path = graph_dir/"video_index.parquet", graph_dir/"concept_video_edges.parquet"
    if vi_path.exists() and cv_path.exists():
        vi = pd.read_parquet(vi_path).copy(); vi[["video_id","ccid"]] = vi[["video_id","ccid"]].astype(str)
        v2c = dict(zip(vi.video_id, vi.ccid)); c2i = {v2c[v]:item2idx[v] for v in item2idx if v in v2c}
        cv = pd.read_parquet(cv_path).copy(); cv[["concept_id","ccid"]] = cv[["concept_id","ccid"]].astype(str)
        cv = cv[cv.ccid.isin(c2i)].drop_duplicates(["ccid","concept_id"])
        concepts = sorted(cv.concept_id.unique()); con2idx = {x:i+1 for i,x in enumerate(concepts)}
        for ccid,g in cv.groupby("ccid"):
            ids = [con2idx[x] for x in g.concept_id.iloc[:cfg.max_concepts_per_video]]
            item_concepts[c2i[str(ccid)], :len(ids)] = ids
    else:
        vi = pd.DataFrame(columns=["video_id","ccid"])

    course_path = graph_dir/"course_video_edges.parquet"
    if course_path.exists():
        course = pd.read_parquet(course_path).copy()
        if "course_id" not in course:
            alias = next((x for x in ("course","courseid","courseId","id_course") if x in course), None)
            if alias: course = course.rename(columns={alias:"course_id"})
        if "video_id" not in course and "ccid" in course and len(vi):
            ccid_to_video = {}
            for video_id, ccid in zip(vi.video_id, vi.ccid):
                if video_id in item2idx: ccid_to_video.setdefault(str(ccid), str(video_id))
            course["video_id"] = course.ccid.astype(str).map(ccid_to_video)
        if {"video_id","course_id"}.issubset(course.columns):
            course[["video_id","course_id"]] = course[["video_id","course_id"]].astype(str)
            course = course[course.video_id.isin(item2idx)].drop_duplicates("video_id")
            courses = sorted(course.course_id.unique()); course2idx = {x:i+1 for i,x in enumerate(courses)}
            for r in course.itertuples(): item_course[item2idx[r.video_id]] = course2idx[r.course_id]

    # IMPORTANT: targets never enter training negative-sampling exclusion.
    train_positive = {u:set(items) for u,items in train_h.items()}
    train_eval_users = sorted(u for u,x in train_h.items() if len(x) >= 2)
    train_eval_h = {u:x[:-1] for u,x in train_h.items() if len(x) >= 2}
    train_eval_target = {u:train_h[u][-1] for u in train_eval_users}
    return dict(train=train, valid=valid, test=test, train_h=train_h, valid_h=valid_h, test_h=test_h,
                valid_target=valid_target, test_target=test_target, eval_users=eval_users,
                train_eval_users=train_eval_users, train_eval_h=train_eval_h, train_eval_target=train_eval_target,
                train_positive=train_positive, n_items=n_items, n_users=len(user_ids), item_ids=item_ids,
                user_ids=user_ids, item_concepts=item_concepts, item_course=item_course,
                n_concepts=len(concepts), n_courses=len(courses), audits=audits)

class PrefixDataset(Dataset):
    def __init__(self, histories, cfg):
        self.histories, self.cfg = histories, cfg
        self.examples = [(u,t) for u,h in histories.items() for t in range(1,len(h))]
    def __len__(self): return len(self.examples)
    def __getitem__(self, ix):
        u,t = self.examples[ix]; h = self.histories[u]
        return torch.tensor(u), torch.tensor(left_pad(h[:t], self.cfg.max_len)), torch.tensor(h[t])

class NARM(nn.Module):
    def __init__(self, n_items, cfg):
        super().__init__(); d=cfg.hidden_dim; self.cfg=cfg
        self.item_emb=nn.Embedding(n_items+1,d,padding_idx=0)
        self.gru=nn.GRU(d,d,batch_first=True)
        self.a1=nn.Linear(d,d,bias=False); self.a2=nn.Linear(d,d,bias=False); self.av=nn.Linear(d,1,bias=False)
        self.proj=nn.Linear(2*d,d); self.drop=nn.Dropout(cfg.dropout); self.scale=math.sqrt(d)
    def encode(self,seq):
        mask=seq.ne(0); lengths=mask.sum(1).clamp_min(1); x=self.drop(self.item_emb(seq)); out,_=self.gru(x)
        # Sequences are left padded, so the most recent event is always last.
        last=out[:,-1]
        alpha=self.av(torch.sigmoid(self.a1(out)+self.a2(last).unsqueeze(1))).squeeze(-1)
        alpha=torch.softmax(alpha.masked_fill(~mask,-1e9),1); local=(alpha.unsqueeze(-1)*out).sum(1)
        return self.proj(torch.cat([local,last],-1))
    def candidates(self): return self.item_emb.weight[1:]

class BERT4Rec(nn.Module):
    def __init__(self,n_items,cfg):
        super().__init__(); d=cfg.hidden_dim; self.cfg=cfg; self.mask_id=n_items+1
        self.item_emb=nn.Embedding(n_items+2,d,padding_idx=0); self.pos=nn.Embedding(cfg.max_len,d)
        layer=nn.TransformerEncoderLayer(d,cfg.heads,4*d,cfg.dropout,batch_first=True,norm_first=True,activation="gelu")
        self.encoder=nn.TransformerEncoder(layer,cfg.layers); self.norm=nn.LayerNorm(d); self.scale=math.sqrt(d)
    def encode(self,seq):
        # Append [MASK] after the history and retain the most recent max_len-1 history items.
        hist=[]
        for row in seq:
            vals=row[row.ne(0)].tolist()[-(self.cfg.max_len-1):]
            hist.append(left_pad(vals+[self.mask_id],self.cfg.max_len))
        x=torch.tensor(hist,device=seq.device); pad=x.eq(0); pos=torch.arange(self.cfg.max_len,device=seq.device)[None]
        z=self.encoder(self.item_emb(x)+self.pos(pos),src_key_padding_mask=pad)
        return self.norm(z[:,-1])
    def candidates(self): return self.item_emb.weight[1:self.mask_id]

class LightGCN(nn.Module):
    def __init__(self,n_users,n_items,edge_u,edge_i,cfg):
        super().__init__(); self.n_users=n_users; self.n_items=n_items; self.layers=cfg.layers; self.scale=math.sqrt(cfg.hidden_dim)
        self.user_emb=nn.Embedding(n_users,cfg.hidden_dim); self.item_emb=nn.Embedding(n_items+1,cfg.hidden_dim,padding_idx=0)
        nn.init.normal_(self.user_emb.weight,std=.1); nn.init.normal_(self.item_emb.weight,std=.1)
        u=torch.as_tensor(edge_u,dtype=torch.long); i=torch.as_tensor(edge_i,dtype=torch.long)
        self.register_buffer("edge_u",u); self.register_buffer("edge_i",i)
        du=torch.bincount(u,minlength=n_users).float().clamp_min(1); di=torch.bincount(i,minlength=n_items+1).float().clamp_min(1)
        self.register_buffer("norm",(du[u].rsqrt()*di[i].rsqrt()))
    def propagate(self):
        u0,i0=self.user_emb.weight,self.item_emb.weight; us=[u0]; its=[i0]; u,i=u0,i0
        for _ in range(self.layers):
            nu=torch.zeros_like(u); ni=torch.zeros_like(i); w=self.norm[:,None]
            nu.index_add_(0,self.edge_u,i[self.edge_i]*w); ni.index_add_(0,self.edge_i,u[self.edge_u]*w)
            u,i=nu,ni; us.append(u); its.append(i)
        return torch.stack(us).mean(0),torch.stack(its).mean(0)
    def encode_users(self,users): return self.propagate()[0][users]
    def candidates(self): return self.propagate()[1][1:]

class HGT(nn.Module):
    """Compact heterogeneous graph transformer over user-item, item-concept and item-course relations."""
    def __init__(self,n_users,n_items,n_concepts,n_courses,train_u,train_i,item_concepts,item_course,cfg):
        super().__init__(); d=cfg.hidden_dim; self.n_users=n_users; self.n_items=n_items; self.layers=cfg.layers; self.scale=math.sqrt(d)
        self.user=nn.Embedding(n_users,d); self.item=nn.Embedding(n_items+1,d,padding_idx=0)
        self.concept=nn.Embedding(max(n_concepts,1)+1,d,padding_idx=0); self.course=nn.Embedding(max(n_courses,1)+1,d,padding_idx=0)
        self.q=nn.ModuleDict({x:nn.Linear(d,d,bias=False) for x in ("user","item","concept","course")})
        self.rel=nn.ParameterDict({x:nn.Parameter(torch.eye(d)) for x in ("ui","iu","ci","ic","oi","io")})
        self.norm_u=nn.LayerNorm(d); self.norm_i=nn.LayerNorm(d); self.drop=nn.Dropout(cfg.dropout)
        edges_u=torch.as_tensor(train_u,dtype=torch.long); edges_i=torch.as_tensor(train_i,dtype=torch.long)
        self.register_buffer("ui_u",edges_u); self.register_buffer("ui_i",edges_i)
        ic_i=[]; ic_c=[]
        for i in range(1,n_items+1):
            for c in item_concepts[i]:
                if c: ic_i.append(i); ic_c.append(int(c))
        io_i=np.flatnonzero(item_course); io_o=item_course[io_i]
        self.register_buffer("ic_i",torch.tensor(ic_i,dtype=torch.long)); self.register_buffer("ic_c",torch.tensor(ic_c,dtype=torch.long))
        self.register_buffer("io_i",torch.tensor(io_i,dtype=torch.long)); self.register_buffer("io_o",torch.tensor(io_o,dtype=torch.long))
    def _aggregate(self,dst_n,src,src_idx,dst_idx,rel):
        out=torch.zeros((dst_n,src.shape[1]),device=src.device,dtype=src.dtype); deg=torch.zeros(dst_n,device=src.device)
        if len(src_idx):
            msg=src[src_idx]@rel; out.index_add_(0,dst_idx,msg); deg.index_add_(0,dst_idx,torch.ones(len(dst_idx),device=src.device))
        return out/deg.clamp_min(1)[:,None]
    def propagate(self):
        u,i,c,o=self.user.weight,self.item.weight,self.concept.weight,self.course.weight
        for _ in range(self.layers):
            mu=self._aggregate(self.n_users,i,self.ui_i,self.ui_u,self.rel["iu"])
            mi=self._aggregate(self.n_items+1,u,self.ui_u,self.ui_i,self.rel["ui"])
            if len(self.ic_i):
                mi=mi+self._aggregate(self.n_items+1,c,self.ic_c,self.ic_i,self.rel["ci"])
            if len(self.io_i):
                mi=mi+self._aggregate(self.n_items+1,o,self.io_o,self.io_i,self.rel["oi"])
            u=self.norm_u(u+self.drop(F.gelu(mu))); i=self.norm_i(i+self.drop(F.gelu(mi)))
        return u,i
    def encode_users(self,users): return self.propagate()[0][users]
    def candidates(self): return self.propagate()[1][1:]

def sample_negatives(users, positives, train_positive, n_items, count):
    rows=[]
    for u,p in zip(users,positives):
        known=train_positive[int(u)]; vals=[]
        while len(vals)<count:
            x=random.randint(1,n_items)
            if x not in known and x!=int(p): vals.append(x)
        rows.append(vals)
    return rows

def metric_row(ranks,top10,targets,users,D,cfg):
    r=np.asarray(ranks); out={"Accuracy@1":float((r==1).mean()),"MRR":float((1/r).mean())}
    for k in cfg.ks:
        hit=r<=k; rec=float(hit.mean())
        out.update({f"Recall@{k}":rec,f"NDCG@{k}":float(np.where(hit,1/np.log2(r+1),0).mean())})
    cr=[]
    for u,recs in zip(users,top10):
        true=set(D["item_concepts"][targets[u]])-{0}; pred=set(D["item_concepts"][np.asarray(recs)].ravel())-{0}
        if true: cr.append(len(true&pred)/len(true))
    out["ConceptRecall@10"]=float(np.mean(cr)) if cr else float("nan")
    return out

@torch.no_grad()
def evaluate_sequence(model,histories,targets,users,D,cfg,device,model_name,split,seed,return_users=False):
    model.eval(); candidates=model.candidates(); ranks=[]; tops=[]; rows=[]
    for start in tqdm(range(0,len(users),cfg.eval_batch_size),desc=f"{model_name} {split}",leave=False):
        us=users[start:start+cfg.eval_batch_size]
        seq=torch.tensor([left_pad(histories[u],cfg.max_len) for u in us],device=device)
        h=model.encode(seq); scores=h@candidates.T/model.scale
        tar=torch.tensor([targets[u]-1 for u in us],device=device)
        for row,u in enumerate(us):
            seen=set(histories[u]); seen.discard(targets[u])
            if seen: scores[row,torch.tensor([x-1 for x in seen],device=device)]=torch.finfo(scores.dtype).min
        ts=scores[torch.arange(len(us),device=device),tar]; rr=((scores>ts[:,None]).sum(1)+1).cpu().numpy()
        rec=(scores.topk(10,1).indices+1).cpu().numpy(); ranks.extend(rr.tolist()); tops.extend(rec.tolist())
        if return_users:
            for u,rank,items in zip(us,rr,rec): rows.append({"Model":model_name,"Seed":seed,"Split":split,"UserIndex":u,"Target":targets[u],"Rank":int(rank),"Top10":" ".join(map(str,items))})
    return metric_row(ranks,tops,targets,users,D,cfg),rows

@torch.no_grad()
def evaluate_graph(model,histories,targets,users,D,cfg,device,model_name,split,seed,return_users=False):
    model.eval(); all_u,all_i=model.propagate(); candidates=all_i[1:]; ranks=[]; tops=[]; rows=[]
    for start in tqdm(range(0,len(users),cfg.eval_batch_size),desc=f"{model_name} {split}",leave=False):
        us=users[start:start+cfg.eval_batch_size]; ut=torch.tensor(us,device=device); scores=all_u[ut]@candidates.T/model.scale
        tar=torch.tensor([targets[u]-1 for u in us],device=device)
        for row,u in enumerate(us):
            seen=set(histories[u]); seen.discard(targets[u])
            if seen: scores[row,torch.tensor([x-1 for x in seen],device=device)]=torch.finfo(scores.dtype).min
        ts=scores[torch.arange(len(us),device=device),tar]; rr=((scores>ts[:,None]).sum(1)+1).cpu().numpy()
        rec=(scores.topk(10,1).indices+1).cpu().numpy(); ranks.extend(rr.tolist()); tops.extend(rec.tolist())
        if return_users:
            for u,rank,items in zip(us,rr,rec): rows.append({"Model":model_name,"Seed":seed,"Split":split,"UserIndex":u,"Target":targets[u],"Rank":int(rank),"Top10":" ".join(map(str,items))})
    return metric_row(ranks,tops,targets,users,D,cfg),rows

def build_model(name,D,cfg,device):
    if name=="NARM": return NARM(D["n_items"],cfg).to(device)
    if name=="BERT4Rec": return BERT4Rec(D["n_items"],cfg).to(device)
    eu=D["train"].u.to_numpy(); ei=D["train"].i.to_numpy()
    if name=="LightGCN": return LightGCN(D["n_users"],D["n_items"],eu,ei,cfg).to(device)
    if name=="HGT": return HGT(D["n_users"],D["n_items"],D["n_concepts"],D["n_courses"],eu,ei,D["item_concepts"],D["item_course"],cfg).to(device)
    raise KeyError(name)

def run_experiments(processed_dir=None, output_dir=None, cfg=CFG):
    processed=locate_processed(processed_dir)
    default="/kaggle/working/four_baselines_correct" if Path("/kaggle/working").exists() else ("/content/four_baselines_correct" if Path("/content").exists() else "./four_baselines_correct")
    out=Path(output_dir or default); (out/"checkpoints").mkdir(parents=True,exist_ok=True); (out/"reports").mkdir(exist_ok=True)
    D=read_splits(processed,cfg); device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    observed=(len(D["train"]),len(D["eval_users"]),D["n_items"])
    expected=(cfg.expected_train_interactions,cfg.expected_evaluation_users,cfg.expected_training_catalogue)
    if cfg.strict_expected_counts and observed!=expected:
        raise RuntimeError(f"Wrong processed dataset/split. Expected train/users/items={expected}, found {observed}. Point PROCESSED_DIR to the BCE-SASRec processed folder used in the paper.")
    print("Processed:",processed); print("Output:",out); print("Device:",device); print("Leakage audits:",D["audits"])
    print({"train_interactions":len(D["train"]),"validation_targets":len(D["valid"]),"test_targets":len(D["test"]),"evaluation_users":len(D["eval_users"]),"training_catalogue":D["n_items"]})
    manifest={"processed":str(processed),"config":asdict(cfg),"audits":D["audits"],"statistics":{"train_interactions":len(D["train"]),"validation_targets":len(D["valid"]),"test_targets":len(D["test"]),"evaluation_users":len(D["eval_users"]),"training_catalogue":D["n_items"]}}
    with open(out/"reports"/"manifest.json","w") as f: json.dump(manifest,f,indent=2)
    prefix_data=PrefixDataset(D["train_h"],cfg); all_results=[]; all_user_rows=[]

    for name in cfg.models:
      for seed in cfg.seeds:
        print(f"\n{'='*72}\n{name} — seed {seed}\n{'='*72}"); seed_all(seed); model=build_model(name,D,cfg,device)
        opt=torch.optim.AdamW(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
        best=-1.; bad=0; epoch_rows=[]; checkpoint=out/"checkpoints"/f"{name}_seed_{seed}.pt"
        if name in ("NARM","BERT4Rec"):
            gen=torch.Generator().manual_seed(seed); loader=DataLoader(prefix_data,batch_size=cfg.batch_size,shuffle=True,generator=gen,num_workers=cfg.num_workers,pin_memory=device.type=="cuda",persistent_workers=cfg.num_workers>0)
        else:
            train_pairs=D["train"][["u","i"]].drop_duplicates().to_numpy(); rng=np.random.default_rng(seed)
        for epoch in range(1,cfg.max_epochs+1):
            model.train(); loss_sum=0.; count=0
            if name in ("NARM","BERT4Rec"):
                iterator=loader
                for users,seq,pos in tqdm(iterator,desc=f"{name} seed {seed} epoch {epoch}",leave=False):
                    users,seq,pos=users.to(device),seq.to(device),pos.to(device)
                    neg=torch.tensor(sample_negatives(users.cpu().tolist(),pos.cpu().tolist(),D["train_positive"],D["n_items"],cfg.negatives),device=device)
                    candidates=torch.cat([pos[:,None],neg],1); h=model.encode(seq); cand=model.candidates()[candidates-1]
                    logits=(h[:,None,:]*cand).sum(-1)/model.scale; loss=F.cross_entropy(logits,torch.zeros(len(seq),dtype=torch.long,device=device))
                    opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),cfg.gradient_clip); opt.step(); loss_sum+=loss.item()*len(seq); count+=len(seq)
            else:
                # Full-batch graph propagation once per epoch. Recomputing the
                # complete graph for every small BPR minibatch is needlessly slow.
                order=rng.permutation(len(train_pairs)); batch=train_pairs[order]
                users=batch[:,0].tolist(); pos=batch[:,1].tolist()
                neg=[x[0] for x in sample_negatives(users,pos,D["train_positive"],D["n_items"],1)]
                ut=torch.tensor(users,device=device); pt=torch.tensor(pos,device=device); nt=torch.tensor(neg,device=device)
                gu,gi=model.propagate(); ps=(gu[ut]*gi[pt]).sum(-1)/model.scale; ns=(gu[ut]*gi[nt]).sum(-1)/model.scale
                loss=-F.logsigmoid(ps-ns).mean()
                opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),cfg.gradient_clip); opt.step()
                loss_sum=loss.item()*len(users); count=len(users)
            evaluator=evaluate_sequence if name in ("NARM","BERT4Rec") else evaluate_graph
            val,_=evaluator(model,D["valid_h"],D["valid_target"],D["eval_users"],D,cfg,device,name,"Validation",seed)
            row={"Model":name,"Seed":seed,"Epoch":epoch,"TrainLoss":loss_sum/max(count,1),**{f"Val_{k}":v for k,v in val.items()}}; epoch_rows.append(row)
            score=val["NDCG@10"]; print(f"{name} seed {seed} epoch {epoch}: loss={row['TrainLoss']:.4f}, val NDCG@10={score:.4f}")
            if score>best+cfg.min_delta:
                best=score; bad=0; torch.save({"state":model.state_dict(),"epoch":epoch,"validation":val},checkpoint)
            else: bad+=1
            if epoch>=cfg.minimum_epochs and bad>=cfg.patience: break
        pd.DataFrame(epoch_rows).to_csv(out/"reports"/f"{name}_seed_{seed}_epochs.csv",index=False)
        saved=torch.load(checkpoint,map_location=device,weights_only=False); model.load_state_dict(saved["state"])
        evaluator=evaluate_sequence if name in ("NARM","BERT4Rec") else evaluate_graph
        for split,h,t,users in (("Train",D["train_eval_h"],D["train_eval_target"],D["train_eval_users"]),("Validation",D["valid_h"],D["valid_target"],D["eval_users"]),("Test",D["test_h"],D["test_target"],D["eval_users"])):
            met,user_rows=evaluator(model,h,t,users,D,cfg,device,name,split,seed,return_users=(split=="Test"))
            all_results.append({"Model":name,"Seed":seed,"Split":split,"BestEpoch":saved["epoch"],**met}); all_user_rows.extend(user_rows)
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    results=pd.DataFrame(all_results); results.to_csv(out/"reports"/"four_baselines_all_metrics.csv",index=False)
    tests=results[results.Split.eq("Test")].copy(); metrics=["Accuracy@1","MRR","NDCG@10","Recall@10","NDCG@20","Recall@20","ConceptRecall@10"]
    summary=tests.groupby("Model")[metrics].agg(["mean","std"]); summary.to_csv(out/"reports"/"four_baselines_test_mean_std.csv")
    pd.DataFrame(all_user_rows).to_csv(out/"reports"/"four_baselines_test_per_user.csv",index=False)
    print("\nThree-seed test results:\n",tests[["Model","Seed",*metrics]].to_string(index=False)); print("\nMean ± standard deviation:\n",summary)
    try:
        import shutil; shutil.make_archive(str(out),"zip",out); print("ZIP:",str(out)+".zip")
    except Exception as exc: print("ZIP warning:",exc)
    return results,summary


In [ ]:
# Start all four baselines. This cell may take several hours.
results, summary = run_experiments(PROCESSED_DIR, OUTPUT_DIR, CFG)


In [ ]:
# Compact final test table
display(results[results['Split'].eq('Test')][
    ['Model','Seed','BestEpoch','Accuracy@1','MRR','NDCG@10','Recall@10','NDCG@20','Recall@20','ConceptRecall@10']
].sort_values(['Model','Seed']))
display(summary)
